In [ ]:
%%capture
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu124
!pip install --upgrade unsloth==2025.9.1
!pip install --upgrade transformers==4.56.1 "huggingface_hub>=0.34.0" "datasets>=3.4.1,<4.0.0"

In [ ]:
import torch
import random
import pandas as pd
from tqdm import tqdm
from huggingface_hub import login
from unsloth import FastLanguageModel
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, StoppingCriteria, StoppingCriteriaList

login('your_huggingface_auth_token_here')

In [ ]:
model, _ = FastLanguageModel.from_pretrained(
    model_name="./models/ViLegalQwen3-1.7B-Base", # or ViLegalQwen2.5-1.5B-Base with Qwen/Qwen2.5-1.5B's tokenizer
    max_seq_length=4096,
    dtype=torch.float16,
    load_in_4bit=True,
    token = "your_huggingface_auth_token_here",

)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B-Base")

In [ ]:
print(f"Model dtype: {next(model.parameters()).dtype}")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    modules_to_save=['embed_tokens', 'lm_head']
)

In [ ]:
df_train = pd.read_csv(r"/.datasets/ViBidLQA/processed/train.csv")
df_val = pd.read_csv(r"/.datasets/ViBidLQA/processed/val.csv")
df_test = pd.read_csv(r"/.datasets/ViBidLQA/processed/test.csv")

In [ ]:
df_train.columns, df_val.columns, df_test.columns

In [ ]:
print(df_train.iloc[0]["instruction"])

In [ ]:
print(df_test.iloc[0]["instruction"])

In [ ]:
dataset_train = Dataset.from_pandas(df_train, preserve_index=False)
dataset_val = Dataset.from_pandas(df_val, preserve_index=False)
dataset_test = Dataset.from_pandas(df_test, preserve_index=False)

In [ ]:
def convert_to_text_format(dataset):
    def map_func(examples):
        return {"text": examples["instruction"]}
    
    return dataset.map(map_func, batched=True, remove_columns=dataset.column_names)

dataset_train_formatted = convert_to_text_format(dataset_train)
dataset_val_formatted = convert_to_text_format(dataset_val)
dataset_test_formatted = convert_to_text_format(dataset_test)

# Simple formatting function
def formatting_func(examples):
    return {"text": examples["text"]}

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_train_formatted,
    eval_dataset=dataset_val_formatted,
    dataset_text_field="text",
    max_seq_length=4096,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2,
    packing=False,
    formatting_func=formatting_func,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=1,
        learning_rate=4e-4,
        fp16=True,
        bf16=False,
        torch_compile=False,
        logging_steps=100,
        eval_strategy="steps",
        eval_steps=100, 
        optim="paged_adamw_32bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none", # Use this for WandB etc
    ),
)

In [ ]:
trainer_stats = trainer.train()

### Inference

In [ ]:
class EndOfConversationCriteria(StoppingCriteria):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.end_token_id = tokenizer.encode("<|im_end|>", add_special_tokens=False)[0]
    
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0][-1] == self.end_token_id

FastLanguageModel.for_inference(model)

stopping_criteria = StoppingCriteriaList([EndOfConversationCriteria(tokenizer)])

In [ ]:
df_test["generated_answer"] = ""

for index, values in tqdm(df_test.iterrows(), total=len(df_test), desc="Generating answer for test set..."):
    instruction = values["instruction"]
    gold_answer = values["answer"]

    inputs = tokenizer(
        instruction,
        return_tensors='pt',
        truncation=True,
        max_length=4096
    ).to('cuda')

    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=4096,
        stopping_criteria=stopping_criteria,
        use_cache=True
    )

    generated_answer = tokenizer.batch_decode(outputs)[0].split("<|im_start|>assistant\n")[1].replace("<|endoftext|>", "")

    df_test.at[index, "generated_answer"] = generated_answer
    
    print(f'==================== Generate output: ====================\n{generated_answer}')
    print(f"==================== Ground truth:====================\n{values['answer']}\n")

In [ ]:
df_test

In [ ]:
df_test = df_test[["answer", "generated_answer"]]
df_test["generated_answer"] = df_test["generated_answer"].apply(lambda x:x.replace("<|im_end|>", ""))
df_test

In [ ]:
df_test.to_csv(r"results.csv", index=False, encoding="utf-8-sig")